In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q /content/drive/MyDrive/mmflood_tiled.zip -d /content/

Entpackt!


In [ ]:
!pip install -q rasterio scikit-learn tqdm

In [ ]:
import numpy as np
import rasterio
from pathlib import Path
from glob import glob
from tqdm.auto import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score
from skimage.filters import threshold_otsu

TILED_ROOT = Path("/content/mmflood_tiled")
F16_EPS = np.finfo(np.float16).eps

def imread(path):
    with rasterio.open(str(path)) as src:
        return src.read()



metrics


In [ ]:
def compute_metrics(y_true, y_pred, ignore=255):
    valid = (y_true != ignore)
    y_t = y_true[valid]
    y_p = y_pred[valid]

    tp = ((y_p == 1) & (y_t == 1)).sum()
    fp = ((y_p == 1) & (y_t == 0)).sum()
    fn = ((y_p == 0) & (y_t == 1)).sum()
    tn = ((y_p == 0) & (y_t == 0)).sum()

    iou_flood = tp / (tp + fp + fn + 1e-7)
    iou_bg    = tn / (tn + fp + fn + 1e-7)
    miou      = (iou_flood + iou_bg) / 2
    prec      = tp / (tp + fp + 1e-7)
    rec       = tp / (tp + fn + 1e-7)
    f1        = 2 * prec * rec / (prec + rec + 1e-7)

    return dict(mIoU=miou, IoU_flood=iou_flood, Prec=prec, Rec=rec, F1=f1)

def evaluate_on_test(predictions_fn, max_tiles=None):
    sar_files  = sorted(glob(str(TILED_ROOT / 'test' / 'sar'  / '*.tif')))
    mask_files = sorted(glob(str(TILED_ROOT / 'test' / 'mask' / '*.tif')))
    dem_files  = sorted(glob(str(TILED_ROOT / 'test' / 'dem'  / '*.tif')))

    if max_tiles:
        sar_files  = sar_files[:max_tiles]
        mask_files = mask_files[:max_tiles]
        dem_files  = dem_files[:max_tiles]

    all_metrics = []
    for sar_p, msk_p, dem_p in tqdm(zip(sar_files, mask_files, dem_files),total=len(sar_files)):
        sar  = imread(sar_p).astype(np.float32)
        mask = imread(msk_p)[0].astype(np.uint8)
        dem  = imread(dem_p).astype(np.float32)
        pred = predictions_fn(sar, dem)
        metrics = compute_metrics(mask, pred)
        all_metrics.append(metrics)

    result = {k: np.mean([m[k] for m in all_metrics]) for k in all_metrics[0]}
    return result

In [ ]:
def otsu_predict(sar, dem):
    vv = np.log10(1 + np.clip(sar[0], 0, None) + F16_EPS)
    vh = np.log10(1 + np.clip(sar[1], 0, None) + F16_EPS)

    thr_vv = threshold_otsu(vv)
    thr_vh = threshold_otsu(vh)

    flood_vv = (vv < thr_vv).astype(np.uint8)
    flood_vh = (vh < thr_vh).astype(np.uint8)

    return (flood_vv & flood_vh).astype(np.uint8)

otsu_metrics = evaluate_on_test(otsu_predict)
for k, v in otsu_metrics.items():
    print(f'{k:12s}: {v:.4f}')

In [ ]:
def extract_features(sar, dem):
    vv = np.log10(1 + np.clip(sar[0], 0, None) + F16_EPS)
    vh = np.log10(1 + np.clip(sar[1], 0, None) + F16_EPS)
    d  = dem[0].copy()
    d  = np.clip(d, -100, 6000)
    ratio = vv / (vh + 1e-7)
    from scipy.ndimage import uniform_filter, generic_filter
    vv_mean = uniform_filter(vv, size=3)
    vh_mean = uniform_filter(vh, size=3)
    vv_std  = np.sqrt(np.maximum(uniform_filter(vv**2, size=3) - vv_mean**2, 0))
    vh_std  = np.sqrt(np.maximum(uniform_filter(vh**2, size=3) - vh_mean**2, 0))
    H, W = vv.shape
    features = np.stack([
        vv, vh, ratio, d,
        vv_mean, vh_mean,
        vv_std, vh_std,
    ], axis=-1).reshape(-1, 8)

    return features

sar_train  = sorted(glob(str(TILED_ROOT / 'train' / 'sar'  / '*.tif')))
msk_train  = sorted(glob(str(TILED_ROOT / 'train' / 'mask' / '*.tif')))
dem_train  = sorted(glob(str(TILED_ROOT / 'train' / 'dem'  / '*.tif')))

#floodratio >2%
X_train, y_train = [], []
used = 0
for sar_p, msk_p, dem_p in tqdm(zip(sar_train, msk_train, dem_train),
                                  total=len(sar_train)):
    mask = imread(msk_p)[0].astype(np.uint8)
    valid = (mask != 255)
    if valid.sum() == 0:
        continue
    flood_ratio = float((mask[valid] > 0).mean())
    if flood_ratio < 0.02:
        continue

    sar = imread(sar_p).astype(np.float32)
    dem = imread(dem_p).astype(np.float32)

    feat = extract_features(sar, dem)
    lab  = mask.flatten()
    valid_flat = valid.flatten()
    feat = feat[valid_flat]
    lab  = lab[valid_flat]
    lab  = (lab > 0).astype(np.uint8)

    if len(feat) > 5000:
        idx = np.random.choice(len(feat), 5000, replace=False)
        feat = feat[idx]
        lab  = lab[idx]

    X_train.append(feat)
    y_train.append(lab)
    used += 1
    if used >= 200:
        break

X_train = np.vstack(X_train)
y_train = np.concatenate(y_train)
print(f"train {X_train.shape[0]} pixel, {used} tiles")
print(f"Flood-%: {y_train.mean()*100:.1f}%")

In [ ]:
#train
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)

feature_names = ['VV', 'VH', 'VV/VH', 'DEM', "VV_mean", "VH_mean", "VV_std", "VH_std"]
importances = rf.feature_importances_
print("feature importances")
for name, imp in sorted(zip(feature_names, importances), key=lambda x: -x[1]):
    print(f'  {name:10s}: {imp:.4f}')

Trainiere Random Forest...
Training fertig!

Feature Importance:
  VH_mean   : 0.2841
  VH        : 0.1802
  VV_mean   : 0.1586
  DEM       : 0.1484
  VH_std    : 0.0946
  VV        : 0.0724
  VV_std    : 0.0388
  VV/VH     : 0.0229


In [ ]:
#eval
def rf_predict(sar, dem):
    feat = extract_features(sar, dem)
    pred = rf.predict(feat)
    H, W = sar.shape[1], sar.shape[2]
    return pred.reshape(H, W).astype(np.uint8)

rf_metrics = evaluate_on_test(rf_predict)
for k, v in rf_metrics.items():
    print(f'{k:12s}: {v:.4f}')

vergleich

In [ ]:
print(f'{"Modell":<25} {"mIoU":>8} {"IoU_flood":>10} {"Prec":>8} {"Rec":>8} {"F1":>8}')

results = {
    'Otsu Thresholding': otsu_metrics,
    'Random Forest':     rf_metrics,
}

for name, m in results.items():
    print(f'{name:<25} {m["mIoU"]:>8.4f} {m["IoU_flood"]:>10.4f} '
          f'{m["Prec"]:>8.4f} {m["Rec"]:>8.4f} {m["F1"]:>8.4f}')

In [ ]:
import joblib
#joblib.dump(rf, '/content/drive/MyDrive/rf_model.joblib')
#print('Gespeichert!')


rf = joblib.load('/content/drive/MyDrive/rf_model.joblib')

## Visualisierung, Beispielpredictions

In [ ]:
import matplotlib.pyplot as plt

sar_files  = sorted(glob(str(TILED_ROOT / 'test' / 'sar'  / '*.tif')))
mask_files = sorted(glob(str(TILED_ROOT / 'test' / 'mask' / '*.tif')))
dem_files  = sorted(glob(str(TILED_ROOT / 'test' / 'dem'  / '*.tif')))

#tiles mit flut
for i, (s, m, d) in enumerate(zip(sar_files, mask_files, dem_files)):
    mask = imread(m)[0]
    if (mask == 1).sum() > 1000:
        idx = i
        break

sar  = imread(sar_files[idx]).astype(np.float32)
mask = imread(mask_files[idx])[0].astype(np.uint8)
dem  = imread(dem_files[idx]).astype(np.float32)

otsu_pred = otsu_predict(sar, dem)
rf_pred   = rf_predict(sar, dem)

vv = np.log10(1 + np.clip(sar[0], 0, None) + F16_EPS)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
axes[0].imshow(vv, cmap='gray')
axes[0].set_title('SAR (VV)')
axes[1].imshow(mask, cmap='Blues', vmin=0, vmax=1)
axes[1].set_title('Ground Truth')
axes[2].imshow(otsu_pred, cmap='Blues', vmin=0, vmax=1)
axes[2].set_title('Otsu')
axes[3].imshow(rf_pred, cmap='Blues', vmin=0, vmax=1)
axes[3].set_title('Random Forest')

for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/baseline_predictions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import json

viz_info = {
    'idx': idx,
    'sar_path': str(sar_files[idx]),
    'mask_path': str(mask_files[idx]),
    'dem_path': str(dem_files[idx]),
}
with open('/content/drive/MyDrive/viz_tile_info.json', 'w') as f:
    json.dump(viz_info, f)
print(f"{viz_info['sar_path']}")